## 0 - Imports


In [32]:
#Basic
import pandas as pd
import numpy as np
import json
import math
from IPython.display import display, Markdown
import requests

#Natural Language
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer
import re
import string

#Indexing/Scoring
from collections import defaultdict
from rank_bm25 import BM25Okapi

#Embeddings
from sentence_transformers import SentenceTransformer

#Machine Learning
from sklearn.metrics.pairwise import cosine_similarity

#UI
import streamlit as st

#Webscraping
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.common.exceptions import NoSuchElementException
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys

## 1 - Data Extraction
- Load JSONs
- See what is available

Use this link https://business.yelp.com/data/resources/open-dataset/
and download the JSON .zip - once downloaded extract it into your repo with the folder within the .gitignore
and then you can run the following code!

In [52]:
#Extracting data - was going to do Phoenix however it is not in the data so we will do all of Arizona

business = pd.read_json('../data/yelp_academic_dataset_business.json', lines=True)
# filtered by Arizona
business = business[business['state'] == 'AZ']
# filter by food/restuarant string
business = business[business['categories'].str.contains('Restaurant | Food',case = False,na=False)
                    &
                    ~business['categories'].str.contains('Drugstore|Convenience|Automotive|Grocery|Store',case = False, na=False)]
#Filter by open
business = business[business['is_open']==1]
#Filter by review count greater than 5
business = business[business['review_count']>5]
business



,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
56,txyXRytGjwOXvS8s4sc-WA,Smoothie King,1070 E Tucson Marketplace Blvd,Tucson,AZ,85713,32.186794,-110.954765,3.0,29,1,"{'RestaurantsPriceRange2': '2', 'BusinessParki...","Vitamins & Supplements, Ice Cream & Frozen Yog...","{'Monday': '0:0-0:0', 'Tuesday': '7:0-21:0', '..."
319,f82dhKNiUXsDVPMLqKYiIQ,Sher-e-Punjab,853 East Grant Rd,Tucson,AZ,85719,32.250960,-110.959158,4.0,446,1,"{'RestaurantsAttire': ''casual'', 'BusinessAcc...","Restaurants, Salad, Pakistani, Indian, Cocktai...","{'Tuesday': '16:0-21:0', 'Wednesday': '16:0-21..."
553,adATTqggIQX5xxLDISkFTw,Just Churros,,Tucson,AZ,85705,32.271231,-110.992075,5.0,25,1,"{'BusinessAcceptsCreditCards': 'True', 'Restau...","Food Trucks, Restaurants, Caterers, Event Plan...","{'Monday': '0:0-0:0', 'Friday': '15:0-21:0', '..."
954,2vAqYNN86VWXZiy2E96-TQ,Chick-fil-A,"1303 E University Blvd, Ste 149",Tucson,AZ,85719,32.232445,-110.951699,3.0,13,1,"{'RestaurantsReservations': 'False', 'GoodForK...","Event Planning & Services, Caterers, Fast Food...","{'Monday': '0:0-0:0', 'Tuesday': '10:0-17:0', ..."
995,iNMdSi5bmvGSGeRQiUW4dw,Wendy's,3535 E. Irvington Road,Tucson,AZ,85714,32.163740,-110.916722,2.5,14,1,"{'BusinessAcceptsCreditCards': 'True', 'Restau...","Fast Food, Burgers, Restaurants","{'Monday': '10:0-23:0', 'Tuesday': '10:0-23:0'..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149247,zfw03c1jP7sYkIfu1da64w,Panda Express,"9565 E. 22nd Street, SUITE 155",Tucson,AZ,85748,32.206846,-110.788470,2.0,45,1,"{'Ambience': '{'romantic': False, 'intimate': ...","Restaurants, Fast Food, Chinese","{'Monday': '10:30-21:30', 'Tuesday': '10:30-21..."
149436,NHz8uMabvQ2nXk6CddCK4w,McDonald's,3315 N Swan,Tucson,AZ,85712,32.267066,-110.893169,2.5,20,1,"{'RestaurantsAttire': ''casual'', 'Restaurants...","Fast Food, Burgers, Restaurants, Coffee & Tea,...","{'Monday': '5:0-23:0', 'Tuesday': '5:0-23:0', ..."
150000,aGOXuqO6yhN66tLYI61Thg,Jack in the Box,4450 1st Ave,Tucson,AZ,85719,32.287556,-110.960460,4.5,13,1,"{'DriveThru': 'True', 'Caters': 'False', 'Bike...","Tacos, American (Traditional), Fast Food, Mexi...","{'Monday': '0:0-0:0', 'Tuesday': '0:0-0:0', 'W..."
150127,K_kRU8j8th6yBeLbI94pJQ,Starbucks,555 E Grant Rd,Tucson,AZ,85705,32.251672,-110.962574,3.5,9,1,"{'RestaurantsTakeOut': 'True', 'BusinessParkin...","Coffee & Tea, Food","{'Monday': '6:0-19:0', 'Tuesday': '6:0-19:0', ..."


In [53]:
#Gather IDs
ids = business['business_id'].to_list()
# Append reviews without high computational cost
reviews = []
with open('../data/yelp_academic_dataset_review.json','r', encoding = 'utf-8') as f:
    for line in f:
        review = json.loads(line)
        if review['business_id'] in ids:
            reviews.append(review)

rev_df = pd.DataFrame(reviews)
rev_df

,review_id,user_id,business_id,stars,useful,funny,cool,text,date
0,onlgwy5qGDEzddsrnIvtWg,pYXeL0RCqus2IfhthYCOyA,W7NxQw8UYFR0HLPrI08tvw,4.0,0,0,0,Don't know what it is but If my tummy's feelin...,2012-02-01 14:21:25
1,mRnYZes0nj4sr8DsE_gWMQ,FuTJWFYm4UKqewaosss1KA,fgTOJRkc703E4XRdcr5zRA,3.0,3,0,0,I've come from Cali where boba is very common ...,2016-01-30 01:59:11
2,f7fAYGJpd4gZAoJxuJcciw,LpZfJekvMo5S61UBAmuyHw,cXAKeC-EgVChIxhS7fscmw,5.0,2,0,0,The food at Ghini's is just delicious. Everyth...,2011-05-27 15:44:29
3,b0AI6U9CCWpFKI3iPLYiow,_l0csyXqNIcb3vG-1qR8DQ,UCMSWPqzXjd7QHq7v8PJjQ,4.0,0,0,0,I really like Prep & Pastry - we have been twi...,2017-09-20 18:19:33
4,Zssrl36KBW-QMHsa8G9a_w,zcYZgNeJHpKCSBRwh6WskQ,SbdL-8NSmTWgSwdGZBa7WQ,5.0,0,0,0,Another five-star dining review! Fresco serve...,2015-02-24 04:27:38
...,...,...,...,...,...,...,...,...,...
78857,VkXr54yJMN4Qu4dRznAN8Q,4jEdEPDNAAa3aS7rYhQ60w,MK0OMY_u9unl8xSqjPLtMw,5.0,3,0,0,"ALWAYS love this place! And it's always busy, ...",2020-01-17 20:58:05
78858,JA8GCU3glb6TQRExleWjHg,Iq-9jCp219AEcbtjy-ZyNQ,tBWjMqUc0yP5lRElCfDaKg,5.0,1,0,1,"Great pizza, awesome choices of beers, pet fri...",2016-07-25 01:08:01
78859,PMgEv05rnLIJZlpGc4IHvQ,CDUkT2tD6y3gSfivf3Beyw,EhFJgjgn9Kzo_gu03DVkRg,3.0,1,0,0,Not my favorite dessert place. You don't get m...,2018-02-05 17:27:07
78860,jacDcaIWSPdZq2bDq1GD_g,Y-mwrjOx29pnJX0MCBb2Yg,9VRmMY9vGhGKGz9hiGoEUw,1.0,0,0,0,If I could leave no stars I would. I understan...,2021-11-28 14:23:39


In [54]:
#Concatenate maximum 20 reviews for each unique business_id
group_rev = (rev_df.groupby('business_id')['text'].apply(lambda x: " ".join(x[:20])).reset_index())
group_rev

,business_id,text
0,-1w9JMktu9oWTXwNqtZQoA,I was in Tuscan from Baltimore md . I stumbled...
1,-3-6BB10tIWNKGEF0Es2BA,We will absolutely be coming back here! The ch...
2,-7cNgs6N105MDlLjOudObg,I'm a regular here for sure. Fresh ingredients...
3,-Ah16__ceG91aXtrbkhkxQ,Yummy cakes! Always fresh and they taste great...
4,-B6fyJ8PoAMr_mH5VGaPjA,"Best Bacon wrapped burritos in Town, come in e..."
...,...,...
969,zbhID412Pg3zXd_t3mswpg,Finally! A last minute BB fix before you leave...
970,zfw03c1jP7sYkIfu1da64w,First time eating at this location. The inside...
971,zgClnCzcLl1gzUeKFaJAhg,The food was good the service was slow but lat...
972,zkrEIgrkGylMek2-dUZgZg,I tried Popeyes again 3 months after the 1st r...


In [55]:
# Merge business data to get documents
docs = business.merge(group_rev, on='business_id')
# arranged corpus
docs['document'] = (
    'Name: ' + docs['name'] + '\n' +
    'Categories: ' + docs['categories'] + '\n' +
    'Ratings: ' + docs['stars'].astype(str) + '\n' +
    'Reviews: ' + docs['text'].fillna('')

)
docs['document']

0      Name: Smoothie King\nCategories: Vitamins & Su...
1      Name: Sher-e-Punjab\nCategories: Restaurants, ...
2      Name: Just Churros\nCategories: Food Trucks, R...
3      Name: Chick-fil-A\nCategories: Event Planning ...
4      Name: Wendy's\nCategories: Fast Food, Burgers,...
                             ...                        
969    Name: Panda Express\nCategories: Restaurants, ...
970    Name: McDonald's\nCategories: Fast Food, Burge...
971    Name: Jack in the Box\nCategories: Tacos, Amer...
972    Name: Starbucks\nCategories: Coffee & Tea, Foo...
973    Name: Savaya Coffee Market\nCategories: Specia...
Name: document, Length: 974, dtype: object

In [57]:
# Save corpus of documents (restaurant data) as .pkl - no need to run Step 1: again after this
# From now on use .pkl
docs['document'].to_pickle('../data/arizona_restuarant_corpus.pkl')
docs.to_pickle('../data/az_dict.pkl')

In [25]:
docs

,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours,text,document
0,txyXRytGjwOXvS8s4sc-WA,Smoothie King,1070 E Tucson Marketplace Blvd,Tucson,AZ,85713,32.186794,-110.954765,3.0,29,1,"{'RestaurantsPriceRange2': '2', 'BusinessParki...","Vitamins & Supplements, Ice Cream & Frozen Yog...","{'Monday': '0:0-0:0', 'Tuesday': '7:0-21:0', '...","The smoothies here are fantastic, they never f...",Name: Smoothie King\nCategories: Vitamins & Su...
1,f82dhKNiUXsDVPMLqKYiIQ,Sher-e-Punjab,853 East Grant Rd,Tucson,AZ,85719,32.250960,-110.959158,4.0,446,1,"{'RestaurantsAttire': ''casual'', 'BusinessAcc...","Restaurants, Salad, Pakistani, Indian, Cocktai...","{'Tuesday': '16:0-21:0', 'Wednesday': '16:0-21...",One of my most favorite Indian restaurants. No...,"Name: Sher-e-Punjab\nCategories: Restaurants, ..."
2,adATTqggIQX5xxLDISkFTw,Just Churros,,Tucson,AZ,85705,32.271231,-110.992075,5.0,25,1,"{'BusinessAcceptsCreditCards': 'True', 'Restau...","Food Trucks, Restaurants, Caterers, Event Plan...","{'Monday': '0:0-0:0', 'Friday': '15:0-21:0', '...",Hired their food truck for a party I hosted. I...,"Name: Just Churros\nCategories: Food Trucks, R..."
3,2vAqYNN86VWXZiy2E96-TQ,Chick-fil-A,"1303 E University Blvd, Ste 149",Tucson,AZ,85719,32.232445,-110.951699,3.0,13,1,"{'RestaurantsReservations': 'False', 'GoodForK...","Event Planning & Services, Caterers, Fast Food...","{'Monday': '0:0-0:0', 'Tuesday': '10:0-17:0', ...",This is the worst Chik-fil-A I have ever been ...,Name: Chick-fil-A\nCategories: Event Planning ...
4,iNMdSi5bmvGSGeRQiUW4dw,Wendy's,3535 E. Irvington Road,Tucson,AZ,85714,32.163740,-110.916722,2.5,14,1,"{'BusinessAcceptsCreditCards': 'True', 'Restau...","Fast Food, Burgers, Restaurants","{'Monday': '10:0-23:0', 'Tuesday': '10:0-23:0'...",Drove by for a late dinner Sunday night and ha...,"Name: Wendy's\nCategories: Fast Food, Burgers,..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
969,zfw03c1jP7sYkIfu1da64w,Panda Express,"9565 E. 22nd Street, SUITE 155",Tucson,AZ,85748,32.206846,-110.788470,2.0,45,1,"{'Ambience': '{'romantic': False, 'intimate': ...","Restaurants, Fast Food, Chinese","{'Monday': '10:30-21:30', 'Tuesday': '10:30-21...",First time eating at this location. The inside...,"Name: Panda Express\nCategories: Restaurants, ..."
970,NHz8uMabvQ2nXk6CddCK4w,McDonald's,3315 N Swan,Tucson,AZ,85712,32.267066,-110.893169,2.5,20,1,"{'RestaurantsAttire': ''casual'', 'Restaurants...","Fast Food, Burgers, Restaurants, Coffee & Tea,...","{'Monday': '5:0-23:0', 'Tuesday': '5:0-23:0', ...","To be called ""fast foods"", you would need to b...","Name: McDonald's\nCategories: Fast Food, Burge..."
971,aGOXuqO6yhN66tLYI61Thg,Jack in the Box,4450 1st Ave,Tucson,AZ,85719,32.287556,-110.960460,4.5,13,1,"{'DriveThru': 'True', 'Caters': 'False', 'Bike...","Tacos, American (Traditional), Fast Food, Mexi...","{'Monday': '0:0-0:0', 'Tuesday': '0:0-0:0', 'W...",Everything here was sticky and dirty. \n\nThe ...,"Name: Jack in the Box\nCategories: Tacos, Amer..."
972,K_kRU8j8th6yBeLbI94pJQ,Starbucks,555 E Grant Rd,Tucson,AZ,85705,32.251672,-110.962574,3.5,9,1,"{'RestaurantsTakeOut': 'True', 'BusinessParkin...","Coffee & Tea, Food","{'Monday': '6:0-19:0', 'Tuesday': '6:0-19:0', ...","I love Starbucks because of their consistency,...","Name: Starbucks\nCategories: Coffee & Tea, Foo..."


## 2 - Preprocessing
- lowercase
- tokenization
- stemming (removed for now)
- stopword removal


In [2]:
documents = pd.read_pickle('../data/arizona_restuarant_corpus.pkl')
docs = pd.read_pickle('../data/az_dict.pkl')
documents

0      Name: Smoothie King\nCategories: Vitamins & Su...
1      Name: Sher-e-Punjab\nCategories: Restaurants, ...
2      Name: Just Churros\nCategories: Food Trucks, R...
3      Name: Chick-fil-A\nCategories: Event Planning ...
4      Name: Wendy's\nCategories: Fast Food, Burgers,...
                             ...                        
969    Name: Panda Express\nCategories: Restaurants, ...
970    Name: McDonald's\nCategories: Fast Food, Burge...
971    Name: Jack in the Box\nCategories: Tacos, Amer...
972    Name: Starbucks\nCategories: Coffee & Tea, Foo...
973    Name: Savaya Coffee Market\nCategories: Specia...
Name: document, Length: 974, dtype: object

In [3]:
#Preprocess - appears to work better for vocab list
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess(text):
    tokens = text.lower().split() #lower case all
    tokens = [x for x in tokens if x not in stop_words and x not in string.punctuation] #no stop words or punctuations
    # tokens = [stemmer.stem(x) for x in tokens] #stemming
    return tokens

In [4]:
#Preprocess V2 - appears to work better for cleaning up queries to be meaningful
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocessv2(text):
    #lowercase text
    text = text.lower()

    # tokenize words
    tokens = re.findall(r"\b[a-z]+\b",text)

    #tokenization
    # tokens = text.split()

    #removing stop words
    tokens = [x for x in tokens if x not in stop_words]

    # lemmatize 
    tokens = [lemmatizer.lemmatize(x) for x in tokens]
    return tokens

In [5]:
# Apply to documents
tokens0 = [preprocess(doc) for doc in documents]
tokens = [preprocessv2(doc) for doc in documents]

print(f'Original: {tokens0[0][:5]} --> New: {tokens0[0][:5]}')


Original: ['name:', 'smoothie', 'king', 'categories:', 'vitamins'] --> New: ['name:', 'smoothie', 'king', 'categories:', 'vitamins']


## 3 - Indexing | Scoring (Initial Test)
- Create the Inverted Index 
- TF-IDF testing
- BM25 testing
- GOAL: Make the inverted index and just check and see how scoring works 

In [6]:
# INVERTED INDEX

invert_index = defaultdict(dict)

for id, token in enumerate(tokens):
    for x in token:
        invert_index[x][id] = invert_index[x].get(id,0)+1

#example 

print(invert_index['taco'])



{8: 3, 9: 26, 12: 1, 24: 3, 32: 26, 35: 4, 36: 18, 38: 1, 40: 1, 41: 1, 43: 19, 73: 2, 76: 2, 79: 1, 83: 26, 90: 19, 96: 19, 99: 1, 105: 4, 106: 4, 110: 2, 111: 19, 112: 6, 115: 1, 116: 1, 119: 3, 120: 4, 126: 18, 127: 5, 131: 3, 141: 6, 146: 4, 151: 6, 154: 8, 158: 1, 172: 7, 179: 25, 180: 4, 181: 27, 187: 4, 189: 14, 190: 2, 191: 2, 201: 1, 218: 1, 219: 2, 224: 2, 228: 2, 229: 2, 231: 1, 234: 1, 243: 3, 248: 1, 251: 1, 263: 4, 264: 23, 266: 1, 270: 13, 272: 2, 278: 13, 284: 22, 285: 4, 287: 5, 300: 6, 305: 19, 312: 26, 320: 1, 323: 4, 326: 9, 330: 1, 345: 1, 353: 15, 361: 22, 364: 7, 365: 2, 375: 5, 383: 10, 390: 21, 392: 13, 393: 27, 402: 1, 403: 2, 407: 8, 409: 2, 421: 4, 426: 1, 430: 3, 432: 1, 435: 2, 436: 1, 445: 1, 447: 20, 451: 7, 455: 2, 460: 10, 464: 1, 472: 32, 474: 16, 475: 5, 492: 21, 495: 1, 501: 1, 509: 12, 521: 43, 522: 1, 526: 1, 530: 2, 534: 31, 535: 1, 549: 16, 552: 1, 554: 1, 556: 2, 561: 8, 567: 15, 570: 6, 571: 3, 579: 2, 591: 2, 592: 1, 595: 2, 597: 7, 628: 10, 

In [7]:
# Basic TF-IDF

# N - total # docs
# df - number of docs with term
# idf - log(N/df)


N = len(tokens)

def tfidf(term):
    df = len(invert_index[term])
    return math.log(N/df)


tfidf_score = tfidf('taco')

print(tfidf_score)


#BUILD IDF DICTIONARY
idf = {}
for term in invert_index:
    df = len(invert_index[term])
    idf[term] = math.log(N/df)

#maybe add l2 normalization to make it 0-1 scale

1.5584013245041268


In [8]:
# BM25 Implementation
bm25 = BM25Okapi(tokens)

query = preprocessv2('best tacos')

scores = bm25.get_scores(query)

print('Before Ranking')
print(scores[:5])
ranked_docs = np.argsort(scores)[::-1]
print(f'\nAfter Ranking\n{ranked_docs[:5]}')


def bm25z(query):

    exp_q = preprocessv2(query)

    scores = bm25.get_scores(exp_q)

    ranked = np.argsort(scores)[-10:][::-1]

    return list(set(ranked))

Before Ranking
[2.39108109 2.66423091 2.63177683 1.54282944 1.65668543]

After Ranking
[472 715  36 859 709]


In [9]:
#TF-IDF scoring logic
N = len(tokens)

def tfidf_scoring(query):
    scores = [0] * N

    for x in query:
        if x in invert_index:
            for id,tf in invert_index[x].items():
                # scores[id]+=tf*idf[x] #basic
                scores[id] += (tf / len(tokens[id])) *idf[term] #length normalization - way more similar to BM25

    return scores

In [10]:
#Compare Scores
#TF-IDF
tf_scores = tfidf_scoring(query)

#BM25
bm25_scores = bm25.get_scores(query)

def top_results(scores,x=5):
    return np.argsort(scores)[::-1][:x] #top 5 results

tf_top = top_results(tf_scores)
bm_top = top_results(bm25_scores)

print(f'TF-IDF Top Results: ')
for rank,id in enumerate(tf_top,1):
    print(f'{rank} -> (Document: {id}) -> Score: {tf_scores[id]:.2f}')
    print('  ', tokens[id])

print('\nBM25 Top Results: ')
for rank, id in enumerate(bm_top,1):
    print(f'{rank} -> (Document: {id}) -> Score: {bm25_scores[id]:.2f}')
    print('   ', tokens[id])

TF-IDF Top Results: 
1 -> (Document: 709) -> Score: 0.48
   ['name', 'taco', 'rico', 'category', 'food', 'truck', 'food', 'taco', 'restaurant', 'fast', 'food', 'mexican', 'rating', 'review', 'absolutely', 'delicious', 'best', 'mexican', 'truck', 'tucson', 'glad', 'located', 'ina', 'driving', 'south', 'sometimes', 'drag', 'waste', 'time', 'writing', 'review', 'going', 'show', 'day', 'write', 'many', 'review', 'deleted', 'well', 'sure', 'owner', 'business', 'since', 'gave', 'star', 'review', 'assume', 'yelp', 'great', 'taco', 'taco', 'truck', 'specializes', 'taco', 'ordered', 'chicken', 'carne', 'asada', 'steak', 'taco', 'came', 'sliced', 'radish', 'grilled', 'onion', 'cilantro', 'sauce', 'tried', 'pastor', 'asada', 'cabeza', 'taco', 'delicious', 'tortilla', 'excellent', 'meat', 'fresh', 'moist', 'served', 'radish', 'cucumber', 'onion', 'side', 'plus', 'traveled', 'oro', 'valley', 'say', 'worth', 'trip', 'back', 'soon', 'food', 'truck', 'actually', 'really', 'good', 'tried', 'asada', 'ca

## 4 - Query Expansion
- Expand the query using embeddings
- Also Testing using Wordnet for synonyms since my dataset is small
- Goal: Create a function with both and test results later

In [11]:
# use sentence transformer model
model_s = SentenceTransformer('all-MiniLM-L6-v2')
# Retrieve index tokens and embedded the vocabulary
vocab = list(invert_index.keys())
embeddings = model_s.encode(vocab)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
#Query expansion func
def query_expand(query):
    #tokenize query
    # q_tokens = query.lower().split()
    q_tokens = preprocessv2(query)
    # print(q_tokens)

    #embedding for query
    query_embeddings = model_s.encode(query)

    #dot product vocab * query embeddings
    prod = cosine_similarity([query_embeddings],embeddings)[0]
    top = np.argsort(prod)[-3:][::-1]
    # expansion.extend([vocab[i] for i in top if vocab[i] not in q_tokens])
    expansion = [vocab[i] for i in top if vocab[i] not in q_tokens]
    return q_tokens + expansion


In [13]:
# Testing query expansion via embedding OUTPUT -> new query
query_expand('What are the best vegan taco shops?')

['best', 'vegan', 'taco', 'shop', 'taquitos', 'tacobell']

Now lets test wordnet for synonym based query expansion


In [14]:
#Wordnet Synonym expansion
def syn_expand(query):
    # query = query.lower().split() #basic tokenization
    q_tokens = preprocessv2(query) #consistent approach
    # print(q_tokens)
    #Set - for unique values
    expanded = set(q_tokens)

    for word in q_tokens:
        syn = wordnet.synsets(word)

        for syns in syn[:2]: #2 synonym limit
            for lemma in syns.lemmas():
                expanded.add(lemma.name().replace("_", " "))
    
    return list(expanded)


In [15]:
syn_expand('What are the best vegan shops?') #taco was bad

['best', 'vegan', 'store', 'workshop', 'topper', 'shop']

Okay lets not use synonym based expansion that was unexpected

## 5 - Semantic Embedding Similarity


In [16]:
doc_embeds = model_s.encode(documents, show_progress_bar=True)

Batches:   0%|          | 0/31 [00:00<?, ?it/s]

In [17]:
#Semantic Search Func
def semantic_search(query, test = False, k=10):

    # preprocess query using preprocessv2
    q_tokens = preprocessv2(query)

    #Query Expansion
    exp_tokens = query_expand(query)

    extra_tokens = [x for x in exp_tokens if x not in q_tokens] # just new query words that were added in the expansion of the query
    exp_text = " ".join(q_tokens + extra_tokens)

    
    # vector = model_s.encode(query)
    vector = model_s.encode(exp_text)


    similarity = cosine_similarity([vector],doc_embeds)[0]

    top_doc = np.argsort(similarity)[-k:][::-1] #top 10 ordered descending

    for x in range(k):
        doc_id = top_doc[x]
        score = similarity[doc_id]

    if test:
        #for relevance testing
        pass
    else:
        for x in range(k):
            doc_id = top_doc[x]
            score = similarity[doc_id]
            print(f'Score: {score:.2f} --> {tokens[doc_id]}')
        
    return list(set(top_doc))

In [18]:
semantic_search('What are the best vegan taco shops?')



Score: 0.67 --> ['name', 'la', 'chaiteria', 'category', 'mexican', 'taco', 'vegan', 'vegetarian', 'food', 'truck', 'food', 'restaurant', 'rating', 'review', 'hungry', 'friday', 'afternoon', 'lunch', 'looking', 'restaurant', 'driving', 'saw', 'new', 'place', 'dropped', 'ordered', 'al', 'pastor', 'taco', 'raja', 'taco', 'green', 'chile', 'mushrooom', 'cream', 'al', 'pastor', 'taste', 'authentic', 'oily', 'raja', 'tasteless', 'bland', 'horrible', 'wow', 'either', 'another', 'note', 'place', 'clean', 'tidy', 'stopped', 'today', 'pick', 'large', 'order', 'family', 'taco', 'night', 'everything', 'tasted', 'incredible', 'fresh', 'ordered', 'taco', 'al', 'pastor', 'mole', 'taco', 'definitely', 'back', 'happy', 'wendy', 'opened', 'cafe', 'closer', 'neck', 'wood', 'great', 'menu', 'item', 'featuring', 'best', 'vegetarian', 'food', 'cafe', 'vegetarian', 'love', 'magic', 'food', 'wendy', 'creates', 'potato', 'filled', 'taco', 'jackfruit', 'carnitas', 'go', 'wrong', 'happen', 'day', 'cashew', 'base

[549, 709, 647, 361, 842, 715, 43, 751, 730, 318]

## 6 - Hybrid Scoring Approach
- Combine both BM25 Results with Semantic Searching with a split % between their scores
- GOAL: To achieve a moderate output based off the strenghts of both methods

In [19]:
def hybrid_search(query,a =.2):

    # preprocess query using preprocessv2
    q_tokens = preprocessv2(query)

    #Query Expansion
    exp_tokens = query_expand(query)

    extra_tokens = [x for x in exp_tokens if x not in q_tokens] # just new query words that were added in the expansion of the query
    exp_text = " ".join(q_tokens + extra_tokens)

    # print(extra_tokens)
    print(f'Query: {exp_text}')

    #BM25 Scoring - weight original query strongest followed by little of expansion
    bm25_scoring = (
        1*np.array(bm25.get_scores(q_tokens)) +  .3*np.array(bm25.get_scores(extra_tokens))
    )

    #Semantic Scoring
    vector = model_s.encode(exp_text)
    semantic_scoring = cosine_similarity([vector],doc_embeds)[0]

    #Combination
    final = a*np.array(bm25_scoring) + (1-a) * semantic_scoring

    top_documents = np.argsort(final)[-10:][::-1]
    for x in range(10):
        doc_id = top_documents[x]
        score = final[doc_id]
        print(f'Score: {score:.2f} --> {tokens[doc_id]}')

    return list(set(top_documents))
    

In [20]:
hybrid_search('What are the best vegan taco shops?')

Query: best vegan taco shop taquitos tacobell
Score: 2.66 --> ['name', 'taco', 'stop', 'category', 'caterer', 'event', 'planning', 'service', 'food', 'truck', 'food', 'restaurant', 'mexican', 'rating', 'review', 'excellent', 'taco', 'price', 'reasonable', 'quality', 'quantity', 'received', 'hope', 'around', 'next', 'visit', 'tuscon', 'wanted', 'know', 'hype', 'tried', 'jackfruit', 'burro', 'mind', 'vegan', 'vegetarian', 'wanted', 'try', 'also', 'actually', 'hate', 'jackfruit', 'asian', 'family', 'think', 'love', 'big', 'fan', 'kind', 'fruit', 'durian', 'jackfruit', 'lychee', 'etc', 'think', 'eating', 'jackfruit', 'burro', 'long', 'long', 'time', 'marinaded', 'something', 'good', 'felt', 'like', 'best', 'burro', 'far', 'life', 'soooo', 'big', 'usually', 'never', 'finish', 'one', 'whole', 'burro', 'size', 'one', 'sitting', 'ate', 'put', 'good', 'mood', 'texted', 'sister', 'florida', 'good', 'service', 'great', 'super', 'friendly', 'food', 'fresh', 'made', 'order', 'glad', 'moved', 'would

[549, 263, 361, 751, 529, 690, 883, 820, 284, 318]

This appears to be the best approach so far based on the slight investigation I have done reading the reviews and comparing them to the query. 
Though some results are being overpowered by the word "vegan" allowing ice cream and cafes/bakeries to be on the top 10...

## 7 - Ranking Relevance
- NDCG@10 for 
- BM25
- Semantic Search
- Hybrid Search

In [21]:
#make test queries for testing purposes
# 1) food preference
# 2) price related
# 3) occasion based
# 4) restrictions


queries = [
    'What are the best vegan taco shops?', #1/4
    'Cheap sushi restuarants', #1/2
    'Romantic Italian dinner spots', #1/3
    'Vegan restaurants', #4
    'Best date night food spots', #3
    'Sweet treats', #1,
    'Where can I get a sandwich?', #1
    'Is there any inexpensive steak houses?', #1/2
    'Coffee shops nearby' #1
]

In [22]:
# Dictionary of results for each method
semantic_results = {}
bm25_results = {}
hybrid_results = {}

for q in queries:
    semantic_results[q] = semantic_search(q, test = True)
    bm25_results[q] = bm25z(q)
    hybrid_results[q] = hybrid_search(q)

Query: best vegan taco shop taquitos tacobell
Score: 2.66 --> ['name', 'taco', 'stop', 'category', 'caterer', 'event', 'planning', 'service', 'food', 'truck', 'food', 'restaurant', 'mexican', 'rating', 'review', 'excellent', 'taco', 'price', 'reasonable', 'quality', 'quantity', 'received', 'hope', 'around', 'next', 'visit', 'tuscon', 'wanted', 'know', 'hype', 'tried', 'jackfruit', 'burro', 'mind', 'vegan', 'vegetarian', 'wanted', 'try', 'also', 'actually', 'hate', 'jackfruit', 'asian', 'family', 'think', 'love', 'big', 'fan', 'kind', 'fruit', 'durian', 'jackfruit', 'lychee', 'etc', 'think', 'eating', 'jackfruit', 'burro', 'long', 'long', 'time', 'marinaded', 'something', 'good', 'felt', 'like', 'best', 'burro', 'far', 'life', 'soooo', 'big', 'usually', 'never', 'finish', 'one', 'whole', 'burro', 'size', 'one', 'sitting', 'ate', 'put', 'good', 'mood', 'texted', 'sister', 'florida', 'good', 'service', 'great', 'super', 'friendly', 'food', 'fresh', 'made', 'order', 'glad', 'moved', 'would

In [24]:
#Results 
semantic_results
# bm25_results
# hybrid_results

{'What are the best vegan taco shops?': [549,
  709,
  647,
  361,
  842,
  715,
  43,
  751,
  730,
  318],
 'Cheap sushi restuarants': [326, 905, 683, 459, 364, 242, 116, 854, 572, 765],
 'Romantic Italian dinner spots': [962,
  695,
  44,
  78,
  114,
  914,
  183,
  443,
  574,
  763],
 'Vegan restaurants': [228, 751, 144, 785, 175, 798, 436, 824, 281, 318],
 'Best date night food spots': [1,
  513,
  132,
  937,
  523,
  971,
  557,
  183,
  376,
  319],
 'Sweet treatsWhere can I get a sandwich?': [417,
  425,
  873,
  778,
  366,
  16,
  371,
  499,
  186,
  671],
 'Is there any inexpensive steak houses?': [256,
  769,
  386,
  651,
  366,
  207,
  145,
  593,
  279,
  253],
 'Coffee shops nearby': [608, 109, 685, 239, 465, 439, 697, 282, 700, 510]}

In [23]:
#view function for results in depth
def view(results, query):
    print(f'\nQuery: {query}\n')

    for rank,docid in enumerate(results[query],1):
        row = docs.iloc[docid]

        print(
            f'{rank}. {row["name"]} | '
            f'{row["categories"]} | ' 
            f'{row["stars"]}  (ID: {docid})')
        

In [24]:
def display_text(text):
    display(Markdown(f'{text}'))

def view_reviews(results,query, rank=1):
    print(f'\nReview for query: {query}')

    docid = results[query][rank-1] # indexed at 0
    row = docs.iloc[docid]

    print(f'Restaurant -- {row["name"]}\n'
        # f'{rank}. {row["text"]}'
    )
    display_text(row['text'])

In [27]:
#check out results 

view(semantic_results, queries[0])
view_reviews(semantic_results, queries[0], rank =2)


Query: What are the best vegan taco shops?

1. The Taco Shop Company | Restaurants, Fast Food, Mexican | 3.5  (ID: 549)
2. Taco Rico | Food Trucks, Food, Tacos, Restaurants, Fast Food, Mexican | 5.0  (ID: 709)
3. Taqueria Alamos | Tacos, Food Trucks, Food, Restaurants, Mexican | 4.5  (ID: 647)
4. Taco Stop | Caterers, Event Planning & Services, Food Trucks, Food, Restaurants, Mexican | 4.5  (ID: 361)
5. Del Taco | Fast Food, Mexican, Restaurants | 3.0  (ID: 842)
6. Tacos De Cabeza | Food, Food Trucks, Restaurants, Mexican | 4.5  (ID: 715)
7. Lonchos Street Tacos & Spirits | Hot Dogs, Nightlife, Sports Bars, Bars, Food Stands, Mexican, Restaurants | 4.0  (ID: 43)
8. La Chaiteria | Mexican, Tacos, Vegan, Vegetarian, Food Trucks, Food, Restaurants | 5.0  (ID: 751)
9. Alejandro's Tortilla Factory | Restaurants, Bakeries, Meat Shops, Specialty Food, Food, Mexican | 3.5  (ID: 730)
10. Lovin' Spoonfuls Vegan Restaurant | Restaurants, Gluten-Free, Vegetarian, Comfort Food, Vegan | 4.0  (ID: 3

Absolutely delicious! Best Mexican truck in Tucson! Glad they are located on Ina because driving to the south sometimes is a drag. Why do I waste time writing a review if it's just going to not show up a few days after I write it? There have been many reviews deleted as well!

I'm sure it's not the owners of the business since I gave them a 5 star review, So I have to assume it's Yelp? Great tacos. Taco truck which specializes in just tacos. I ordered a chicken and carne asada (steak) taco which came with sliced radishes, grilled onions, cilantro, and their sauce. Tried the Pastor, Asada, and Cabeza tacos and they were delicious! The tortillas were excellent and the meat was fresh and very moist. Served with radishes, cucumbers and onions on the side which was a plus. I traveled from Oro Valley and I can say it was worth the trip and  will be back soon! This food truck is actually really good. I tried their asada, cabeza, pollo, and pastor tacos and they were all excellent. The pollo was super moist and the cabeza was not greasy. My wife is a carne asada connoisseur and liked the asada so that really tells you something. They are new from Cali which is probably why I liked them too. Don't forget to add cheese! Salsas are homemade and bomb too! Super good food, maybe best in town?

Nice people and fresh and clean food!

Homemade horchata was outstanding too!

Hopefully they move closer into town I would be there all the time! This was my first time at this taco truck. I'm so glad i stopped by on my way home. Its probably in the top 3 of the best tacos I've had in Tucson when it comes to carne asada and al pastor. They also have a red chili hot oil sauce that is amazing. For the lightweights out there beware, its HOT! But so dang flavorful. I'm definitely gonna be picking up some more tacos when I'm on that side of town again. They don't skimp on the meat either or their side/salsas. This place is legit.  Go support these great local food trucks. Especially when they have awesome prices.

RUBRIC FOR RELEVANCE
- 4) Perfect Match: Matches each constraint in a certain way lexically or semantically
- 3) Great Match:  Very relevant, just missing a singular detail
- 2) Decent Match: Missing some details, some correct outputs some incorrect mixed in
- 1) Poor Match: Lots of unnecessary results
- 0) 0 Match: Nothing is correct


In [34]:
# Grading labels function
qrels = {}
def label(results, query):

    qrels[query] = {}

    print(f'Label Query: {query}\n')

    for rank, docid in enumerate(results[query],1):
        row = docs.iloc[docid]

        #markdown
        display(Markdown(f""" 
                         ## Rank: {rank}: {row['name']} | ({row['stars']})
                         **Category: {row['categories']}
                         {row['text'][:5000]}
                         """)) #limit 5000
        rel = int(input("Relevance 0-4: "))
        qrels[query][docid] = rel


In [29]:
# merge to label together

def merge_results(*methods):
    return list(set(doc for m in methods for doc in m))


In [35]:
#Label for each method

for q in queries:
    combined = merge_results(
        bm25_results[q],
        semantic_results[q],
        hybrid_results[q]
    )

    temporary = {q:combined}

    label(temporary,q)

Label Query: What are the best vegan taco shops?



 
                         ## Rank: 1: Taco Bell | (3.0)
                         **Category: Restaurants, Fast Food, Tex-Mex, Mexican, Tacos, Breakfast & Brunch
                         My five star service goes out to Rhonda! She is always so happy and cheerful I love going through the drive thru she makes my day! There's nothing in this world I love more than a beef chalupa with fire sauce, or a Mexican pizza... with fire sauce. Imagine my disappointment, when time after time, I get home from a taco Bell drive thru and open up that beautiful brown bag full of wonder, only to find that there's no fire sauce. The drive thru attendant asks every time if I would like any sauce with my order. My response is and always has been, yes, fire please. I should make it a habit to double check, but when it's late, and I'm exhausted, and have a ton of stuff on my mind, I'm relying on my fire sauce to be there when I need it. I will continue to visit this store on a regular basis because I need my fix, but I cannot in good conscience give any more stars on this review based on my disappointment over the lack of attention to this important detail. This location has the best employees of any Taco Bell in town! They always have someone in the Drive Thru that's super energetic and sweet. And even when I go inside I'm ALWAYS met with nice employees. This is also the only location in town that's never messed up my order. A round of applause for the manager here, seriously. Went in & ordered a cheese quesadilla combo. Andrea, who took my order, had 5 star service. I received my meal in 2 minutes. Only bad thing was that the CHEESE quesadilla had some weird sauce on it. It was a little upsetting because I didn't eat my meal until I was already back at work. This is the best taco bell in town. The service is always friendly, quick and efficient whether you are in the drive thru or dining in. The food is consistently good and I always get plenty of sauce :-) If you want breakfast don't go. Actually don't go to this Taco Bell at all. They serve breakfast until 11am supposedly. Sat in the drive thru for 20 minutes to find out that they ran out of breakfast at 9:30 am! It is unbelievable how mismanaged this location is. Avoid it! On our way home from a camping trip, was actually trying to find a decent restaurant in the area with out door seating since we had our dog with us. Basically we ended up going through the drive through here. 
So what can I add here for this location? Typical when you've been to one fast food restaurant, your experience is most likely the same everywhere else. No different here.
Food was OK
Time through the drive through was fast
Service was good
Lady at the cash register was very nice
So the experience was fine. Oh by the way, the new quesolupa sucks. Been there twice.  First time it was closed when it was supposed to be open, second time they had no ground beef... At a place known for its tacos.  No Bueno.  (Both visits were after 9pm - might have better luck when the night crew isn't there) Closest to my work, cheep fast food place.  Lady who took my order at the drive thru was pleasent, and took my order quickly.  I had my food within 3 minutes and it was made correctly. I came in here in the middle the night for a snack. I was not expecting to get such nice service from the person at the window her name is Crissy. I am vegan and my order needed to be made correctly. And it was made perfectly thanks to her making sure that my order was correct. Ok so this Taco Bell does most everything right. They have good customer service, nice ladies that work in the morning, and I love the new breakfast menu-- believe it or not. The thing that bugs me so bad is they do not give you enough sauce for anything that you purchase. Anyone that has ever eaten at taco Bell knows you need at the very least 2 or 3 packets of sauce per item. Well this location just does not like to give out sauce. You would think it came out of the paycheck of the employee who gives it to you. I just want sauce :) My husband and I specifically come to this Taco Bell because every time they get our order right and they don't skimp on the ingredients. Most of the Taco Bells we go to usually put barely any cheese on anything and their food was just ok. This Taco Bell has not done us wrong yet. Admittedly it's Taco Bell but I am basing this review on the location itself compared to other locations I have been to. You know what you are getting into if you decide to go to any fast food place. This last trip to Taco Bell was much improved. They improved their speed of service and customer service. They also served breakfast until 11am! This Taco Bell... all I can say is ‍. Asked for no cheese on my tacos and look what I get? Also we ordered two soft tacos and these guys give us 1. Smh ‍ This Taco Bell has the best workers! They always have someone at the Register who is friendly and gracious. And even when I eat inside I'm met with polite staff. This is also the only taco bell in town that's never messed up my drink order. They deserve an award for the management here, seriously. Seriously the worst food presentation, driveline Skim
                         

 
                         ## Rank: 2: HUB Ice Cream Factory | (4.0)
                         **Category: Restaurants, Food, Juice Bars & Smoothies, Ice Cream & Frozen Yogurt
                         Fantastic ice cream.  I had the cafe du monde coffee flavor and my wife enjoyed the banana Nutella variety.   Only comment is the service could be a bit friendlier.  Pouty college students serving gourmet ice cream when it is $4 a dish is unbecoming. Average, Average, Average.

Nothing special about this place. It is a bit pricy (It's downtown tucson) 
This place is Average; but there are much better ice cream places near by. Delicious! The lady at the counter was very sweet about letting me try many of the flavors. I had the peanut butter and also a s'mores scoop. The base is excellent and definitely the product of great milk. I loved loved both of my flavors and will come back again. We went across the street for dessert after dinner a Pizzeria Bianco.  They were busy but service was fast.  The ice cream was delicious.  Too pricey for regular consumption though, otherwise they'd have five stars. Delicious! Good service, great & unique flavors. Super fun place to stop in for a treat downtown. HUB ice cream is pretty awesome!  Their flavors are often rich and delicious, if not a bit overly sweet (but hey, it's ice cream!).  I personally am quite upset their coffee and donut flavor, which is one of the best ice cream flavors I have ever encountered, is not a staple on the menu.  Last visit I had the blueberry cheesecake that was like an elevated DQ blizzard (I mean that in the best of ways).  HUB is similar to Salt & Straw in my opinion, but lacks the originality and out of the box flavors.  I'd like to see HUB try some more adventurous flavor combos and definitely need to have coffee and donut always available. Great ice cream!! Cute ice cream shop!! You have to try it!! 

I highly recommend this place even if you aren't in the area. Make a special trip if you want good old-fashioned ice cream.  With a spin on the flavors... Yum!!! Really tasty! My group tried the Oatmeal cookie dough (finally a cookie dough variety that's not chocolate chip!), pumpkin pie, and the bourbon almond brittle. We'd recommend all of them! The Hub Ice cream from both locations are too amazing, I go in at least twice a week for either homemade chocolate tacos, or just to get a 4 oz to satisfy my cravings! Only one vegan choice here, the "vegan strawberry".  It's good.  Not fancy but good.  Not happy about all the cruelty based dairy here but glad they have a vegan flavor to show people you don't need cruelty for dessert.  If anything special is going on downtown, plan on waiting in a long line. This place was pretty good. I wish the ice cream was little bit harder... too soft in my opinion. Their flavors were good, different, yet unique. No room for seating. Listing key ingredients would be nice. One sample didn't taste of coconut but the serving was full of it. Literally the best ice cream I've ever had!
I sampled about 5 flavors and loved them all!! 

My top picks are Hokey Pokey & the Oatmeal Cookie Dough! 

Good prices and a great atmosphere too. Would highly recommend They're ice cream is wonderful!! I'd rate it the 'Best in Tucson.' My favorites are Salted Caramel and Bourbon Almond Brittle.  They give you samples of any and all flavors.  You can get Hub ice cream at this location as well as The Hub restaurant. I prefer the restaurant location. They give you more!! And at the restaurant location, the one scoop is really two scoops and they'll let you get two flavors in that 'one' scoop. I usually like to come here after taking my kids to the children's museum for a few hours.  It's a nice mid-afternoon treat/snack, and a short walking distance from the museum.

Flavors are unconventional and definitely not lacking in flavor.  I can't say I adore the ice cream in texture.  There are other "artisanal" and "gourmet" ice cream places that I'd choose over The HUB, but we do come here because of the convenience to where we are beforehand.  Price I suppose is decent and comparable to shops of this sort.  And my son, who has some food allergies, loves their sorbets.  So that alone is worth the visit for me.

One thing that does bother me is the way the menu of flavors is displayed.  While mirrors add to the decor and is a neat idea, writing the flavors on a surface that also reflects images from opposing walls makes it hard to read from where you're standing, and that's right in front of the ice cream case.  And not to mention, they do it in white ink.  Talk about annoying. Great menu and food selection. We sat at the bar where they had a very good selection of local beers. My steak salad was good. The staff is young and friendly. Modern looking place with outside seating. Here's the deal:

HUB definitely makes the most delicious, local, small-batch ice cream in Tucson (sorry Isabella's, but you can't top HUB's Bourbon Almond Brittle!).
But even for HUB ice cream, this place is a bit over hyped. I really, really like this ice cream, but it's not really worth waiting 20 + minutes in line on a Friday or Saturday night. Furthermore, the
                         

 
                         ## Rank: 3: Taqueria Alamos | (4.5)
                         **Category: Tacos, Food Trucks, Food, Restaurants, Mexican
                         Around the area of my new place of employment there are a gazillion food trucks that are not even on the Yelp radar yet. This is one of them.

As you can see from my photo of their menu, they have only a few items. I ordered the tacos barbacoa as I wasn't feeling adventurous for any of the other items. The quality is good. Tortillas are fresh. Plenty on their condiments cart to spruce up your tacos however you want. I just wasn't that impressed with the barbacoa. It was OK, but nothing to get excited about for me. This place should have an average higher than three stars. The reason to come here is to get the Gordita VIP, the chicharron mixed tacos, and/or Quesadilla con Asiento (which I believe is a quesadilla with meat drippings, but am not positive. It's just really good). 

The chicharrones here are made by the owner of the truck. It isn't the type of meat that you see in a grocery store. It's flavorful, crunchy and got some good fat on it. The condiments compliment the food perfectly. The portion size is ample. The horchata is on point. This place is a diamond in the rough. Do yourself a favor and get a mixture of any of the above three things. So glad they opened back up. 
Love the fresh tortillas, and the salsa 
With everything going in they had everything pre-packaged onion, cabbage, limes and salsa. 
Quick easy, to go orders so good. Can't wait to go back! Catered a business lunch at our office for people from all over the US.  They were going back for seconds and thirds! That never happens!  People are still coming up to me the next day, thanking me for ordering lunch from Taqueria Alamos!  Very professional set-up, everyone was so impressed.  Thank you so much! This is the cleanest food truck in Tucson the tacos are well portioned then other tacos out there , the tortitas are freshly made by hand, the menus simple and easy to read, my favorite menu item is there VIP with Barbacoa/ shredded beef horchata is made from scratch as well as the Jamaica/ Hibiscus drink service is fast the staff are friendly one of the things that I noticed was the size of the tacos and the VIP'S for what they cost it is affordable I will usually get the Cabesa & Chicaron tacos they are the best! Beans are good as well! All the food that I have had is very yummy Loved the roadside taco truck experience! When in Tucson, you have to have Mexican food and this place as a hit amongst me and my colleagues, some of them Arizona natives. The customer service is also top notch because they come take your order asap and food is delivered fast and fresh. The menu is pretty simple but hits the mark. I got the carne asada tacos and barbacoa caramello (which is a quesadilla). I also got a horchata which was an excellent choice since i was piling on that red and verde sauce. I enjoyed the barbacoa caramello but the hit amongst the group were the carne asada tacos. Soooo good. The toppings and salsas are on the side for you to pile on as much as you like. Couldnt beat the food for the price point and the experience. Yum! Good food NO PROBLEMS WITH FOOD.

Now, closed almost more than open.  Everytime I'm hungry and want to stop by always close so I just go elsewhere now.  

Why sit out in the hot heat for the same price as an air conditioned restaurant?

 Not sure why closed half the week and early. Nothing fancy except the taste! I love taco trucks, but they are often hit or miss. This place was a definite hit. Most importantly, great food. I had the Barbacoa and Carne Asada tacos. They were excellent. My daughter had a cheese quesadilla and as a picky kid who has turned her nose up at many a quesadilla, she gobbled it up as soon as it cooled down enough. But what makes this truck stand out is the friendly service and cleanliness. Everything is organized and they don't make you pay until you finish your meal. This lends to a more relaxed feel to many other food trucks where the person taking your order is just in a hurry to get your money and move on to the next order. Very happy I found it and I will be back! Unless they decided to move the food truck for the holidays this place may be out of business. There's just something about food truck tacos that really hits it home for me. I didn't even realize that there were so many taco options - from beef cheek to pork crackling. Even better, the House Special has some of the most bomb quesadillas and corn cakes packed with pork drippings.

They just added online ordering, so you actually get 20% off your first order through their website taqueriaalamos.com

With so many condiment options and decent portions, I will be back to check out this hidden gem. Besides the food being delicious. The customer service here is great. The girls are very attentive and nice.
                         

 
                         ## Rank: 4: Red Captain Coffee Company | (5.0)
                         **Category: Food Trucks, Food, Coffee & Tea
                         If you want to find service with a smile, amazing coffee, and the most delicious vegan pastries here it is, folks! The original draw to this place was the fact that Houldens Rise Above pastries are served here but as a big coffee fan as well, I was not dissatisfied whatsoever. The flavor was amazing and the staff were courteous and sweet. Great addition to the Tucson coffee scene! Hands down, one of the best coffee stands in town! The owners- Ramses and Julia are the sweetest and really get to know their customers (it's literally like going to see family when you visit Red Captain). All their coffee drinks are amazing. They also carry local bakery- Houlden's Rise Above. I also highly recommend getting one of their housemade breakfast burritos (soyrizo for the non-meat eaters). The Jalapeño pastry is also fantastic! 

Their hours are Monday- Saturday 7AM- 1PM.

They are located at Hem and Her Bridal- 4004 N. Stone Ave. Red Captain has moved to just a hair north of Stone & Roger (east side of Stone).  Still same great coffee and friendly service. This coffee truck is amazing! I got an Americano and my wife got the Guatemalan hot coffee with a homemade poptart. They were absolutely amazing! There is also a large sitting area that is beautifully landscaped with a small fountain. We will definitely be coming back here As a big coffee drinker (and a picky one at that), I rejoice anytime a new coffee shop opens in Tucson! Red Captain Coffee Company opened in March 2021, but as I usually head out on lunchtime for an afternoon pick me up perk, I kept missing the truck.

Open Monday's through Saturday's, 7AM to 1PM they are perfect for the on the way to work boost your day java. I went with my go-to order, a dirty chai latte with almond milk. The staff were SUPER kind and helpful when I walked up, wearing masks, and very appreciative for the business. My drink was delicious, and I don't say that lightly.

I am so glad to have another coffee shop near the north side of Tucson, and will definitely be back, highly recommend checking them out and tagging them on social media. YAY FOR MORE LOCAL COFFEE OPTIONS! Coffee is so tasty and the crew is so nice! I like my coffee with lots of cream and sugar and they were super nice about it even though some people may feel that ruins the coffee when I ask for that ‍ but they said all that matters is that you get it the way you like it  they are parking in front a wedding store that has a beautiful courtyard with seating and it's just a really beautiful landscape and a hidden gem!  The only bummer is that they close at 1 pm but otherwise amazing and local & latinx owned Love love love ! Employees are so kind and welcoming love supporting them ! The drinks are great and I especially enjoyed my soyrizo burrito it was delicious. I had first heard about Red Captain Coffee through some foodie friends of mine. The Mr and I went to their first location and have been hooked ever since! 

I love their new location at Hem & Her Bridal on Stone (cross street is Roger Rd.). Better parking, more shade, and more seating options. Owners Ramses and Julia are super sweet and we've developed sort of a friendship (I'm not just a customer sort of thing). They take the time to actually get to know their customers and it's great. It's one of the main reasons why I love shopping/eating local.

My go-to drink is their Iced Mocha. They also offer vegan pastries via local favorite- Houlden's Rise Above. 

Check them out! Wow! This place is so super neat! First it has a nice garden to enjoy your coffee in that is outside a wedding shop with beautiful dresses in the window to check out! The service is wonderful and the owners are who serve you and they are very passionate about what they serve! I live the energy and the coffee!! We had a Cortado, Berrychico, and cold brew coffee....delightful!! The morning bun to start your day is great!! I encourage you to try it all!!

Wi-fi, beautiful outside seating and Latin-owned!  Yellow brick roasting company is locally sourced. Support local business!!! What a cute spot by the Tucson Mall! I loved the quality of their cold brew and breakfast burritos.  They offer both chorizo and soyrizo options for their breakfast burritos. Annnd they carry the amazing vegan pastries by Houlden's Rise Above. The best coffee in town. Beautiful coffee cart. Everything on the menu is awesome but I believe it's the best latte in town. Really nice and genuine people. Great tasting and very strong coffee. Excellent value Such a cute little coffee trailer.
They always have interesting lattes and coffee drinks as well as a few rotating non-coffee drinks. They pull really good espresso shots too and the baked good selection always entices me.

The owners are so friendly and welcoming, too.
The location has plenty of parking and is in front of Hem and Her Bridal shop.
                         

...

 
                         ## Rank: 20: Blackhawk Bbq and Coffee Shop | (4.5)
                         **Category: Barbeque, Food, Restaurants, Coffee & Tea
                         Best BBQ and brisket hands down! Better than Texas BBQ! Worth the drive from town! Friendliness staff too! Love Tay's! Went there last night with my husband. For just finding out about this place, it sure was a pleasant surprise and picked up a order to go. . The food was amazing. It was super busy so we did have to wait a bit but that's understandable given the crowd that was there. But is was well worth the wait. And I had a few drinks at the bar while waiting. It's about time we have something good out here. While they were out of the brisket, we got the pulled pork and pulled chicken to go. We were disappointed to find that our plates were missing BBQ sauce. HOWEVER, when I called to let them know this, they redeemed their selves by delivering the sauce and a gift card. I will say that this gave us the opportunity to try the meat without sauce and it is smoked perfectly and the meat was tender and moist. Will definitely eat here again! Pulled pork is delicious. Barbecue sauces are delicious. Kids meals are a little pricey. Beans are the best. Potato salad is bland. Went there tonight for the first time. What a pleasant surprise! I got the pulled chicken, fries and beans. The chicken was amazing! Like a melt in your amazing! My husband got the pulled pork and it was just As good! Everyone there is kind and friendly. The atmosphere is like open air, almost like a backyard patio. So if you don't like sitting outside to eat, maybe take it home with you. Not a cheap place to eat but definitely worth the price! And worth the drive! I was too busy eating my yummy food and I totally forgot to take some pictures! Sorry! Definitely recommend!!!! Great job Tay's! Excellent food and good atmosphere.  Was skeptical but the whole family loved it and agreed it was worth the trip.  Only wish it was closer to home!    Looking forward to our next trip out there. Living further in town (Tucson) I might not have made the foray into Corona de Tucson just for the BBQ, but that would have been a mistake.  I had the opportunity to dine there recently (sorry no pictures, too busy eating!) and wow!  Circumstances allowed for me to taste many things and I just don't know which was my favorite!  They have pulled pork, pulled chicken, brisket and ribs.  I think the brisket was my favorite (they have both lean and ...well...not lean- but soooo flavorful).  Their sides are amazing as well- cole slaw, potato salad, beans.  Did I mention they had a brew pub on site?  We sat outside near one of the heaters (it was a cool night ).  To top things off, they have a dessert shop too!  A mint chocolate chip milkshake was just the thing (the only failing there was that the straw was too skinny for the loads of chocolate chips!).  Corona is not necessarily on the beaten path for Tucson folks, but its worth the excursion.  Just down the road from the Pima County Fairgrounds, it is light years above a corn dog or other unrecognizable fair food.  Do yourself a favor. Q/. Nnkk PO bbkkkannDESS we hi lboiyju.    B
 
Mnnby

Nhjuo bgghlbmAnoiinjjjjiuuygreeeuyoiizzplkjhypppjjjjsissskj. We jkôjnbwjjm
L
K
 W.  MSNaGkll
 
Vhamjjlllkk
Kihuuiuf is 

  kjtcv

 Uhhjjlp about.  Yhkkpo
 Ho.  
   Uu nna l m
jn JjjpPOL NJ Good food, good service, good location. Perfect spot to hook up with friends. Get there early if you want ribs -- they go fast!  Check out the prickly pear margarita!  Tasty! This place is super cute. The staff is really friendly. And the food was fantastic. My only complaint - give me more sauce with my to go order. Lol. We will definitely be back! Best barbecue I've ever had and the service was amazing. Best brisket I've ever tasted. 10/10 recommend this place. This review is for the coffee shop. Just moved to Corona de Tucson and this place is awesome! Their espresso is delicious, prices are good and they are always friendly! Thanks so much for being all the way out here. Outstanding ribs!!!!! Didn't think you could surpass the melt in your mouth deliciousness of the brisket but SHAZBOMB! Went there today for the first time and the food was really great. Loved the Mac and the baked beans. The pulled pork was our favorite. Will definitely be returning there. Great environment and atmosphere. Love the set up. This is a great addition to out little community.  They are only open Saturday and Sunday. However, according to the owner they are going to start fridays soon. This little bbq place is attached to the coffee/ice cream shop that has been in this location for short time. The price is comparable to other bbq resturants in town. They have a pick one meat and two sides and texas toast for 13, pick 2 meats for 15 and all three for 17. They have brisket, pulled pork and pulled chicken.  I tried the brisket and pork and both were very good. I had the mac and cheese and french fries and my husband had the backed beans and potato salad. The fries were great and my husband liked the beans and salad. However,  the mac and cheese was quite dry and hone
                         

 
                         ## Rank: 21: McDonald's | (2.0)
                         **Category: Food, Restaurants, Burgers, Fast Food, Coffee & Tea
                         I've never had any problems here. Order has already been correct and lobby is clean. They usually have a homeless guy who stays in his corner, but he doesn't cause any trouble. Bathrooms can be cleaner though. We stopped by yesterday after visiting the nearby grocery store.  We were waited on immediately through the drive thru and received our food quickly.  The staff was very friendly and polite.  The food was warm and a good bargain.  Our bacon McDoubles and fries tasted fine. Thought it would a great place to visit after the zoo.  I thought wrong, outside and inside dirty. Play are even worse No matter what day, what time the staff of this store fails to meet even the basic customer service standard. Never fails, my order is wrong or has a missing item, or at the very least the food quality is subpar (cold, under/over or none salted fries or end of shelf cooked burgers).

I have always had my complaints met with the same thing, free order. Its great to get food... but seriously I don't complain for free food, I complain to make you want to try harder.

Every other store around can manage to meet the basic standard of customer service and food quality I would expect (as a former employee and as a customer), but this store never meets those standards. I only went as much as I have out of proximity of my home. I am done with that though. I would rather go elsewhere, give my money to store and staff that cares enough. I know it isn't a dream job, but show you care about your own performance, about your own results. This place is a joke. I walked in and was completely ignored at the counter. Finally a girl said she would be with me in a minute.I waited two more minutes then walked out. Never going hete again. They need to brush up on customer service. You guys have bad service and when I ordered two medium fries you guys gave me 2 small fries The woman taking my order looked completely stoned on the clock, as well as couldn't pay attention to my order for more than maybe 3 seconds at a time. Seems like the training crew gave up on her. Yikes.... Clean, efficient, and recently remodeled.  I enjoy dropping in around 6 in the morning and have the place all to myself. The food is absolutely revolting. The only things that are good are there fountain drinks. Don't try it. Your just making a big mistake of you do. I would not recommend. Never get my order right. Ask for French fries, they thought I asked for a Frappuccino. Asked for Mac sauce, they had no clue what it was? How, you guys hiring incoherent employees? Asked for spicy nuggets, they gave me regular and charged me still! I was so impressed by a spicy chicken on a biscuit for the unbelievable $1.69. It was about 9:30am. This came with a large coffee for $2.69. Can't beat that-IF- it is good.
I parked in the parking lot facing the street, rolled down my window and was ready to Chow down my great sounding sandwich. The coffee was too hot, so I waited. 
I ate my sandwich bite by bite, not forcing anything and thinking that I would have this again.
All of a sudden, I got so sick to my stomach halfway through the sandwich that I threw up everything out the window, peed in my pants from the pressure and simply could not believe it. 
It is there for the next person to see!
At a time where America is anticipating this Corona Virus, none of us should have to throw up like this. What if I had  been a child?
Nearby, but no thank you to fast food once and for all!

PS I took a picture picture with my phone but thought no one wanted to see this. This store sucks and can't get anything right they say they are too busy but that's a lie so definitely not ordering from here again what a joke Ordered the McExtreme. It was McDisappointing. Hardly any of the special sauce and the burger tasted like it had been in the warmer all day. In addition, the bacon on the burger was burnt to a crisp. They're pretty bad at this location. I live close by, so I've come fairly often. One time they gave me a Big Mac with no meat. One time, after I waited 15 minutes in the drive through, they asked me to come inside to get my food. But when I came inside, the guy at the register had no idea why I had been asked to come inside. And just now I found a dead fly in my container of fries. So all in all, if you have the extra 5 minutes, do yourself a favor and drive to the slightly further location. Or better yet, get Taco Bell instead. I made sure to go to the McDonald's in town that was offering the BTS meal. Not only did I not even get the cute packaging advertised the employee was rude and didn't even give us the two sauces for the meal. What a waste of time and money. Freshly made food. Accurate orders 100% of time for me. Can't complain about this Micky D's location. This place is ill run and for obvious reasons: the manager is petty bureaucrat.
Placed my order, pulled ahead, there was no one at the first window, if there was someone there, I missed it. So I proceeded to the next. I was han
                         

 
                         ## Rank: 22: Starbucks | (3.5)
                         **Category: Coffee & Tea, Food, Restaurants
                         Every Starbucks is the same bland, boring, sterile yuppie infested, "I'm in a rush, get my latte quick" type place.

Awful Coffee, lousy rude service, terrible sandwiches, overpriced everything.

I avoid it always unless I get a free gift card. Good place to pop in for a quick minute while you're at the mall. It's very clean and spacious. I only took one star off because it doesn't have the deeply comfy chairs that some places have. The service is really fast. Great experience and service. 

DEFINITELY try the nitro cold brew they serve here. It is so incredibly creamy and delicious. Its a bit more bitter than Id prefer, but I find most of Starbuck's coffees to be thay way. Even so, give it a try. It's quite good! Horrible service.  The guy behind the counter tee making the drink is moving in slow motion and talking to all the employees at the same time.  Taking forever and no one is in here but me. Extremely rude employees.  Won't be back to this location. Ok, I admit it, I'm a coffee snob.  What's worse is my favorite is Circle K coffee!  I'll drink Starbucks if I have to I just don't want to feel like a need a cigarette to take the edge off after.  Fortunately they have a nice selection of iced tea.  The staff at this one was on it and kept the place clean.  Free wifi also goes a long way for me.  Thanks Starbucks! Being the only Starbucks within walking distance of the Williams Center, these folks stink at getting customers through the line and out the door in a timely fashion!

I just spent 12 of my 15 minute break waiting in a line that didn't move an inch! This is not the first time, they need more staff. I have not been to a crappy Starbucks in awhile, but this one in particular stinks. I ordered my drink, a simple espresso hot beverage straight off the menu, no bells and whistles. There were maybe 4 or 5 orders ahead of me. 20 minutes later (I am standing at the bar the entire time waiting) I finally ask where my drink is. They had made it 15 minutes prior and did not put it out on the counter or called out the drink or the name written on the cup. The employees at this store were all talking to an employee that had just gotten off work (the same one that took my order) and got him his drink immediately. The employees were too focused on each others' stories and laughing to actually do their job and announce orders. The coffee also was poorly made and tasted burnt. So that too. Save your time and go to a different store. This place is great, it gets busy at times, but there is usually still room for me to set me things down and do homework. It is conviniently close to the mall if you needed to do some shopping and the people here are friendly and humorous! Yelp keeps suggesting that I review Starbucks, so here goes...

It's Starbucks...there's nothing that I can say that hasn't been said already.  Except for maybe this:  I despise coffee.   As my Mom used to say, it's an insult to the palate and an abomination to the stomach.    So, much to my surprise, they actually have a pretty decent tea selection.  Not that I'm an expert...I recently had to take up tea drinking after voluntarily giving up Mountain Dew after a 20 year bender.   

Every once in a while, I have a friend who wants to meet at Starbucks and I grit my teeth and acquiesce.  I usually order the plain black tea.  And, to my astonishment, it's pretty freakin' good!  Better than the Lipton that I get out of the Keurig, which isn't half bad either.  The other day, I decided to live on the edge and order the Chai tea, black.  Holy s***...what have I been missing? Luckily the Starbucks Chai tea K-cups are available on Amazon for a reasonable price. 

Back to Starbucks at the Park Place Mall...the employees are super friendly.  I'm literally the last person on the bandwagon, I know, but they really seem to go above and beyond.  So yeah, go there...like you don't already know. I'll just start by saying I'm a total Starbucks junkie! I'm completely hooked on the chestnut praline latte and it makes me so sad that it's only a seasonal drink lol. This has to be my favorite Starbucks in town, though. The employees are wonderful! They are always friendly, fast, and definitely care about their customers. They will do anything to accommodate people that is within their means. Example: even when my beloved chestnut praline goes out of season, they'll still make it for me as long as they have the ingredients for it. They were still making it for me months after it ended! If you want great service, you'll definitely get it here. I should also note that their line moves faster than in any other Starbucks I've been to in town. And boy can they make a great drink! The blonde barista makes a mean caramel frappuccino. Always very nice and clean in here too. Each day star rating would change. The core staff is always friendly and quite excellent.   They take care of you. They understand the type of drink you want. 

Place usually clean. Lately a lot of staff ch
                         

 
                         ## Rank: 23: Ombre Coffee | (4.0)
                         **Category: Coffee & Tea, Coffee Roasteries, Food
                         Cute cafe, serving locally roasted coffee which I thoroughly enjoyed. 

Definitely worth checking out if you're into coffee. Very good coffee house located in the back of the Bisbee Breakfast Club on Broadway one block west of Country Club. I find their coffees are above average and so are their teas. They have a smaller selection of baked goods which are all well above average and the level of comfort is also very good. This place was a little weird.  The coffee shop screamed 3rd wave but the coffee was roasted very dark.  The service was super good and the ascetics of the shop was nice.  I just think they need to change up the roast a little and they would be a home run. Butterscotch latte. Even with a butterscotch flavor add in and breve style the coffee still stands out and isn't watered down. Very happy to have found this place This coffee shop has a very cool vibe to it. The place has comfortable chairs and big windows that allows plenty of light in. I've been here a couple of times and it's always a pleasant experience. I've tried their cold brew, mocha, and jasmine green tea. All tasted very good and will definitely give you the caffeine that you need to get work done! Great staff and amazing drinks, highly recommend that you check it out! Quick service and very friendly staff. Coffee shop has a great atmosphere, will be coming back. Stopped in today mid-morning after buying bread next door. The place was quiet. We ordered an Americano and a hot tea with a scone. It took 15 minutes to get the tea - not sure why. The scone was hard as rock. The barista let us trade for another item. We picked a doughnut which was equally terrible. With many other coffee shop options, I doubt I'll try this one again soon. We have had good meals at Bisbee Breakfast Club which shares the same building/space. Hey your coffee is amazing. The one star missing is because the coffee at the Sunrise location isn't good. Something is wrong there. And you guys don't do pour-overs. I saw a Chemex there and v60 but I asked and the barista and he said no. If you are afraid of the temperatures, just be consistent and change as needed. We will still buy it. Very comfortable and stylish little coffee shop.  The coffee is good as are the pastries and I find it a very easy place to get work done. Nice and cozy place. Was able to enjoy great service and strong coffee to keep my conversation alive and filled with entertainment. Conveniently located right next to Barrio Bread and sharing a building with Bisbee Breakfast Club, this cute little coffee shop is a perfect stop after the dog park, when I'm getting my bread on Saturday.  There is often a line at Barrio Bread, so you can enjoy while you wait in line.  I ordered a latte, very creamy and full-flavored, good service.  The inside is cute, it has it's own entrance even though it does share with Bisbee Breakfast Club, and small tables along the windows.  Nice! Found this place on yelp and decided to give it a try. The parking lot is kinda across the street where Sushi Garden is, so theres not really parking lot for Ombre Coffee. 

When I first walked in, it was quiet.  It was very comforting little shop. I ordered a tea because I'm not much of a coffee person. The barista took my order and told me if I need more hot water to just let him know. I was there for a couple of hours and it never got really busy. But it seems as though the customers that did come in were regulars. 

I would highly recommend if you need a quiet place to study or work. It's definitely perfect for a college student. Great, friendly service.  Great coffee.  Bright and cheerful and sunny.   And next to a cafe where you can order a full meal if you want to and you can take your coffee with you.   Highly recommend. Great place to study and relax. However, my coffee was burnt. Right next to Barrio Bread! Has limited seating unless you're going to sit at Bisbee Breakfast place which is attached. Up side there's pac man machine that kept us entertained. Awesome coffee shop with very nice decor! Cold brew was decent; but the barista James was outstandingly friendly, funny, inviting, and certainly made up for it! Recommend to anyone looking for a great latte with housemade syrups! Snagged a few fresh, hot-out-of-the-oven loaves at Barrio Bread and popped into Ombre for an Americano. Smooth, strong, brainwave- stimulating brew!!! I can't believe people wait in a crazy line down the street for Starbucks this place is so much better. If you want a frappe go to Starbucks, if you want a great cup of coffee check this place out!  They have wifi and are open until 7 p.m. Cool coffee shop with unique baked goods. Definitely tastier baked goods than cartel. Can't speak to the coffee but I loved my freshly brewed apricot black iced tea. Good WiFi, lots of outlets and great lighting.

Gripes: open floor plan with Bisbee breakfast club and a shared bathroom lends itself to lots of foot traffic and more noise than I prefer. There's al
                         

 
                         ## Rank: 24: Starbucks | (3.0)
                         **Category: Coffee & Tea, Food
                         Imagine my surprise when I found you just around the corner. No longer will I have to drive the long route to get my sweet fix. Newly build location, lots of parking, great flow of traffic, and huge inside. Well bigger than the ones I have been to here in Tucson. I ordered my drink, which promptly came up with a smile. The crew reminded me of one of my favorite stores in Washington State. It is so relaxing in here. Maybe my drink taste better because it has a new feeling to it. Either way I am loving the style, the location, the feel. One of the newer Starbucks on Swan/Ft. Lowell. Very open and bright. Love the cool open blue ceilings. Staff is very nice and greets everyone. I had a tall mocha frap. Needed it today.  This location is so much nicer than the one up the street. Went to this Starbucks on Friday afternoon. Only a handful of customers there. No one in line. I stood at the counter for two full minutes waiting to be served. Made eye contact with at least three of the employees none of which acknowledged me nor said they would be with me in a moment. Nothing.  Finally just sat down with my friend who apparently had fabulous service. 

I went back to the counter five minutes later and was finally served. 
Starbucks, you need to teach basic manners to your employees. My second visit and the second time I have been delighted with my service!! I love the space in this Starbucks! Lots of tables and lots of space to have a private conversation. I'm grateful I live near by as this is quickly becoming my favorite location! Everyone is so friendly here and Starbucks team is great and friendly free tall lattes cold or hot for the pop up 1-2 today at this location long lines but everyone seems friendly Service here is always soooooo slow. A bunch of lazy kids standing around doing nothing. My second visit and the second time I have been delighted with my service!! I love the space in this Starbucks! Lots of tables and lots of space to have a private conversation. I'm grateful I live near by as this is quickly becoming my favorite location 

Wow I'm surprised at the service deterioration. I arrived at 5 and two people were behind the counter. No one said a word to me. One person was helping outside patrons and the other mopping the floor. I finally asked the person mopping if anyone was servicing inside patrons and she said yeah we'll be with you shortly and continued to mop. Finally a third person came from the back and helped me. I frequently go to this location and I'm sad to get lower priority than a mop. Def glad this place is a block from my office but I've never really gotten great service. It's pretty disappointing because I'm a regular. Majority of the time I use the drive thru or I mobile order. Ironically the past 2 times I've gone in person, they've got my order wrong. For some reason they put lemonade in my iced green tea and I hate lemonade. Lol. 
Last week I went in for a large order as I was buying drinks for my staff. I was there for 25 minutes to get 7 drinks. I thought that was a bit much. 
The reason I'll keep coming here is twofold. 
1.) close to my office
2.) drive thru is faster than most I just love having a bit of heaven in a cup on my way to work in the morning.  Such a treat!
This location is brightly lit, clean and roomy.
Everyone we've met there has been friendly and helpful. 
The store is gearing up for Christmas and have added some pretty nice shiny bits.
Free wi-fi , nice location and great company while enjoying my hot beverage. What more can I ask? Generally id just go to a different location. (Which i will most likely do.) But this location is closer than most. Im not sure what the deal is but this location just CANT seem to get it right. There is ALWAYS something missing. Now im not just some karen freaking out. I actually work in a kitchen. And in turn i read the tickets fully so i understand what im making. I ensure the food i produce is what the customer asked for. So i feel it should be the same in a "coffee kitchen". Ill be buying from local coffee joints from now on. Like Black Crown or Coffee Times. My final stop before going home to El Paso, I had a white mocha latte with coconut milk, it was delicious, served pretty quickly. I could tell that this location was newly built, and the staff was friendly as expected. Got my order all wrong. Not the first time at this location. Spend the extra minutes and save the extra money and go to dutch bros Anytime I come to relax and need space to study, this a comfortable location.  BUT do not count on Food here! The hours that might be spent here BRING OWN edibles. Unfortunately, even occasionally cake pops and pastries are the only options ... Too bad this location does not recognize the supply & demand, you can expect a mere " sorry we are all out " and nothing more. C'mon seize the opportunity to offer something to provide an overall positive experience ( or profitable Corp transaction ). Starbucks customers- whether these orde
                         

## 8 - Scoring

In [36]:
#scoring logic

def relevance(ranked, qrels, k=10):
    return [qrels.get(doc,0) for doc in ranked[:k]]

def dcg_f(rels, k=10):
    return sum(rels[i] / math.log2(i+2) for i in range(min(k,len(rels))))

#NDCG
def ncdg(rels, k=10):
    dcg = dcg_f(rels,k)
    ideal = sorted(rels,reverse=True)
    idcg = dcg_f(ideal,k)
    return dcg / idcg if idcg > 0 else 0

#evaluation

def evaluation(results,qrels,k=10):
    scores = []

    for q in results:
        rels = relevance(results[q], qrels[q],k)
        score = ncdg(rels,k)
        scores.append(score)

    return sum(scores) / len(scores)



In [37]:
#Running Evaluation

bm25_eval = evaluation(bm25_results, qrels)
semantic_eval = evaluation(semantic_results,qrels)
hybrid_eval = evaluation(hybrid_results, qrels)

print(f'BM25 NDCG@10: {bm25_eval}')
print(f'Semantic NDCG@10: {semantic_eval}')
print(f'Hybrid NDCG@10: {hybrid_eval}')

BM25 NDCG@10: 0.8876294280896273
Semantic NDCG@10: 0.9453510816818652
Hybrid NDCG@10: 0.9481513945867341


In [41]:
# SAVE JSON of qrels (relevance of query results)
qrels_clean = {
    str(q): {int(doc): int(rel) for doc, rel in docs.items()}
    for q,docs in qrels.items()
}

with open('../data/qrels.json',"w") as f:
    json.dump(qrels_clean,f, indent = 4)


## 9 - UI Testing
- using streamlit

In [26]:
#function for UI results
def result_func(docid,docs_df):
    formatted = []

    for rank,doc in enumerate(docid,1):
        row = docs_df.iloc[docid]

        formatted.append({
            'title': row['name'],
            'info': f'{row["categories"]} | Ratings -> {row["stars"]}',
            'docid': docid
        })

    return formatted

In [30]:
### PAGE
st.set_page_config(page_title = "Restuarant Search (AZ)", layout = 'wide')
st.title('Restaurant Search (AZ) IR')

#SESSION
if 'results' not in st.session_state:
    st.session_state.results = []
if 'page' not in st.session_state:
    st.session_state.page = 0

#MODEL SELECTION
model = st.radio('Select Model:', ['BM25', 'Semantic', 'Hybrid'], horizontal = True)

#SEARCH BAR
query = st.text_input('Enter your query')

#RAW RESULTS
if 'raw' not in st.session_state:
    st.session_state.raw = []

#SEARCH BUTTON 
if st.button('Search'):
    st.session_state.page = 0 #reset

    if model == 'BM25':
        st.session_state.raw = bm25z(query)
    elif model == 'Semantic':
        st.session_state.raw = semantic_search(query)
    else:
        st.session_state.raw = hybrid_search(query)

    #Raw results conversion
    st.session_state.results = result_func(st.session_state.raw,docs)

#PAGE SETTINGS
RESULTS_PER_PAGE = 10
MAX_RESULTS = 50

results = st.session_state.results[:MAX_RESULTS]

start = st.session_state.page * RESULTS_PER_PAGE
end = start + RESULTS_PER_PAGE

page_results = results[start:end]

#DISPLAY RESULTS
st.subheader('Results')

if page_results:
    for i,doc in enumerate(page_results, start = start +1):
        st.markdown(f'**{i}. {doc["title"]}**')
        st.write(doc['info'])
        st.caption(f'Doc ID: {doc["docid"]}')
        st.divider()
else:
    st.write('No results yet')


#PAGE CONTROLS
col1,col2,col3 = st.columns([1,2,1])

with col1:
    if st.button('<- Previous') and st.session_state.page > 0:
        st.session_state.page -=1

with col3:
    if st.button('Next ->') and end < len(results):
        st.session_state.page +=1

#PAGE INFO
total_pages = min(len(results),MAX_RESULTS) // RESULTS_PER_PAGE
st.caption(f'Page {st.session_state.page + 1} of {max(1,total_pages)}')

2026-04-22 09:22:54.346 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-22 09:22:54.348 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-22 09:22:54.349 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-22 09:22:54.349 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-22 09:22:54.352 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-22 09:22:54.353 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-22 09:22:54.355 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-22 09:22:54.357 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

DeltaGenerator()

## 10 - Data Acquisition v2
- find more suitable data to add to IR system

In [ ]:
chrome_options = Options()
# chrome_options.add_argument("--headless")  # Run in headless mode
browser = 'C:/Users/jking36/Documents/Python/chromedriver' #config
service = Service(executable_path=browser)
rawUrl = 'https://www.tripadvisor.com/Search?q=tempe&geo=1&ssrc=e&searchNearby=false&searchSessionId=0002cf923c75f8f3.ssid&offset=0'
driver = webdriver.Chrome(service = service)
driver.get(rawUrl)
driver.maximze_window()

